# Async Demo Manual Mode
This notebook runs the async pipeline modules step by step while manually providing the rate data artifact.

In [ ]:
from pathlib import Path

from simkit.config import defaults, schema
from simkit.core.battery_config import ConfigureBatteryModule
from simkit.core.cost_calc import CostCalculatorModule
from simkit.core.perf_sim_simple import SimplePerformanceSimModule
from simkit.core.pipeline import entry_point_validate, exit_point_compose
from simkit.core.project_analyzer import ProjectAnalyzerModule
from simkit.io import readers, writers



In [ ]:
spec_path = Path("simkit/tests/fixtures/pipeline_configs/demo_linear_alt.yaml")
specification = entry_point_validate(spec_path)
entry_spec = specification.modules["entry_point"]
base_dir = specification.source_path.parent if specification.source_path else spec_path.parent

fixture_dir = spec_path.parent.parent

def _resolve_entry_artifact(channel: str) -> Path:
    binding = entry_spec.outputs[channel]
    artifact_path = binding.artifact_path
    if artifact_path is None:
        raise ValueError(f"Entry channel {channel!r} is missing an artifact path")
    return (base_dir / artifact_path).resolve()

geography = readers.read_json_model(_resolve_entry_artifact("geo"), schema.Geography)
load_profile_path = _resolve_entry_artifact("load_profile")
rate_info_manual = readers.read_json_model(
    fixture_dir / "rateinfo_tou_synthetic.json", schema.RateInfo
).model_copy(update={"source": "manual_notebook_override"})
assert len(rate_info_manual.energy_price_usd_per_kwh) == 8760



In [ ]:
load_profile = readers.read_parquet_load_profile(
    load_profile_path, source="manual_notebook"
)
battery_module = ConfigureBatteryModule()
battery_config = battery_module.run(load_profile, rate_info_manual, None).data
cost_module = CostCalculatorModule()
cost_breakdown = cost_module.run(battery_config, geography).data
perf_module = SimplePerformanceSimModule()
telemetry = perf_module.run(battery_config, load_profile, None, rate_info_manual).data
analyzer_module = ProjectAnalyzerModule()
financial_results = analyzer_module.run(
    rate_info_manual, telemetry, defaults.default_financial_params(), cost_breakdown
).data
for series in (telemetry.charge_in_kwh, telemetry.discharge_out_kwh, telemetry.soc_kwh):
    assert len(series) == 8760



In [ ]:
module_versions = {
    "configure_battery": ConfigureBatteryModule.version,
    "cost_calculator": CostCalculatorModule.version,
    "simple_performance_sim": SimplePerformanceSimModule.version,
    "project_analyzer": ProjectAnalyzerModule.version,
}
pipeline_result = exit_point_compose(
    specification=specification,
    rate_info=rate_info_manual,
    battery_config=battery_config,
    cost_breakdown=cost_breakdown,
    telemetry=telemetry,
    financial_results=financial_results,
    module_versions=module_versions,
)
output_dir = Path("notebooks/outputs/manual_mode")
output_dir.mkdir(parents=True, exist_ok=True)
writers.write_json_model(pipeline_result, output_dir / "pipeline_result.json")
writers.write_json_model(rate_info_manual, output_dir / "rate_info_manual.json")
{
    "battery_capacity_kwh": battery_config.capacity_kwh,
    "annual_savings": financial_results.annual_savings,
    "run_description": pipeline_result.pipeline_metadata.run_description,
    "output_folder": pipeline_result.provenance.output_folder,
    "output_dir": str(output_dir),
}

